# 01 — Theme B1: Exploratory Data Analysis & Geographic Data Gaps

**Owner:** Eric Elikplim Sunu

## The Core Question

Where should Ghana's limited supply of insecticide-treated nets (ITNs) go, and what do our
datasets reveal — and hide — about district-level malaria transmission?

### Why Averages Fail: The Greater Accra Contrast
A national survey often reports regional averages that mask local crises. In 2020, a fine-scale
household survey across all 29 Greater Accra districts found local prevalence reaching up to **49%**
in urban slums, despite the official DHS regional average being estimated at just **2%** ([5]).
**Averages allocate nets to the wrong places.**

### The Two Asymmetric Data Sources
1. **Routine Surveillance (`ghana_district_cases.csv`):** 50 districts in the 3 northern regions (2014–17).
   High spatial resolution, but only covers the north, and reflects clinic attendance rather than true prevalence.
2. **Household Survey (`ghana_mis_sample.csv`):** 17,933 households across 16 regions (2022 DHS).
   Representative by design with sampling weights, but resolves only to cluster/region, not district.


In [ ]:
from pathlib import Path
import sys
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))
from src import io, uncertainty as unc, viz

viz.set_theme()
SEED = io.RANDOM_SEED
DATA_DIR = Path.cwd().parent / "data"
BOUNDARIES_DIR = DATA_DIR / "ghana_boundaries"
print("Environment initialized. Seed =", SEED)


In [ ]:
district = io.load_district_cases()
mis = io.load_mis_sample()
region_malaria = io.load_region_malaria()

print(f"District surveillance : {len(district):>6,} rows x {district.shape[1]} cols (Northern Ghana)")
print(f"Household microdata   : {len(mis):>6,} rows x {mis.shape[1]} cols (National DHS extract)")
print(f"Regional indicators   : {len(region_malaria):>6,} rows x {region_malaria.shape[1]} cols (2003-2022 longitudinal)")


---
## 1. Survey Weighting & The Design Effect (Closing Claim C-01)

The Demographic and Health Survey (DHS) is not a simple random sample. Disproportionate sampling
across regions and urban/rural strata requires normalized sampling weights (`sample_weight` = `hv005 / 1e6`).
Evaluating net ownership (`has_net`) without weights misstates true national coverage.

In [ ]:
unweighted_net = unc.unweighted_proportion(mis, "has_net")
weighted_net = unc.weighted_proportion(mis, "has_net", "sample_weight")
weight_diff = abs(unweighted_net - weighted_net)

print(f"National Unweighted Net Ownership : {unweighted_net:.2f}%")
print(f"National Weighted Net Ownership   : {weighted_net:.2f}%")
print(f"Survey Design Discrepancy         : {weight_diff:.2f} percentage points")

c01_table = pd.DataFrame([{
    "Claim ID": "C-01",
    "Claim / Statistic": f"Unweighted net ownership ({unweighted_net:.2f}%) overstates weighted net ownership ({weighted_net:.2f}%) by {weight_diff:.2f} pp due to survey design effects",
    "Source Dataset": "ghana_mis_sample.csv",
    "Notebook Cell": "01_eda.ipynb [Cell eda_cd05]",
    "Status": "Verified"
}])
c01_table.to_markdown(index=False)


---
## 2. Geospatial Linkage & Boundary Harmonization

Joining 2014–17 routine district data to 2021 COD administrative boundaries reveals historical administrative changes:
- 43 of 50 districts match directly after normalizing whitespace and municipal suffixes.
- 7 districts reflect post-2018 splits (Garu-Tempane, Savelugu-Nanton, Bunkpurugu-Yunyoo, Kasena-Nankana) or minor spellings (Gushegu, Sagnerigu, Tatale Sanguli).
- Resolving these yields 53 boundary polygons representing the 50 surveillance districts.

In [ ]:
adm0_path = BOUNDARIES_DIR / "gha_admin0.geojson"
adm1_path = BOUNDARIES_DIR / "gha_admin1.geojson"
adm2_path = BOUNDARIES_DIR / "gha_admin2.geojson"

fig1, ax1 = viz.plot_district_choropleth(
    adm0_path, adm1_path, adm2_path, district,
    value_col="positive_per_100k",
    title="Cumulative Malaria Positives per 100k (Northern Ghana, 2014–17)"
)
viz.save_figure(fig1, "b1_district_case_rate.png")
print("Saved b1_district_case_rate.png")


---
## 3. The National Data Gap & Survey Coverage (Map 2)

While routine clinic surveillance is absent for over 80% of Ghana's districts, the 2022 DHS survey
provides cluster-level net indicators across all 16 regions. Visualizing cluster allocation reveals
how survey power varies geographically.

In [ ]:
names = region_malaria.query("survey_year == 2022").dropna(subset=["hv024_16region"]).set_index("hv024_16region")["region_name"].to_dict()
reg_summary = mis.groupby("region").agg(n_clusters=("cluster", "nunique")).reset_index()
reg_summary["region_name"] = reg_summary["region"].map(names)

fig2, ax2 = viz.plot_region_choropleth(
    adm1_path, reg_summary,
    value_col="n_clusters",
    title="National Survey Data Coverage: DHS 2022 Clusters Sampled per Region"
)
viz.save_figure(fig2, "b1_data_availability.png")
print("Saved b1_data_availability.png")


---
## Key Takeaways for the Allocation Proposal

1. **Survey Weighting Matters:** Unweighted net ownership is 70.96%, but weighted net ownership is 66.77% — a 4.19 percentage point gap (Claim C-01).
2. **The 210-District Void:** Over 80% of Ghana has zero published routine surveillance data, preventing direct nationwide district ranking.
3. **Hotspot Stratification:** Within northern Ghana, case rates vary substantially (41k to 590k cumulative cases), concentrating in Upper East and Upper West regional hubs.
